In [1]:
# Install if needed:
# !pip install pandas scikit-learn nltk matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

nltk.download('stopwords')

# 1. Load dataset
df = pd.read_csv("events_cleaned.csv")
print(df.head())
print(df.columns)

# 2. Select text column
# Change this to your actual text column if needed
text_col = df.select_dtypes(include='object').columns[0]

# 3. Clean text
stop = set(stopwords.words('english'))

def clean(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return ' '.join(w for w in text.split() if w not in stop)

df['clean_text'] = df[text_col].fillna('').apply(clean)

# 4. NLP - TF-IDF
tfidf = TfidfVectorizer(max_features=2000)
X = tfidf.fit_transform(df['clean_text'])

# 5. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = model.fit_predict(X)

# 6. Show clusters
print(df[[text_col, 'cluster']].head(20))

# 7. Important words in each cluster
words = tfidf.get_feature_names_out()

for i in range(k):
    top = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:",
          ', '.join(words[j] for j in top))

# 8. Visualize
pca = PCA(n_components=2)
points = pca.fit_transform(X.toarray())

plt.scatter(points[:,0], points[:,1], c=df['cluster'])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Event Clusters")
plt.show()

# 9. Save result
df.to_csv("events_clustered.csv", index=False)
print("Done! Saved as events_clustered.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_5362/3204706540.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_col = df.select_dtypes(include='object').columns[0]


   event_date       weight_class              fighter1              fighter2  \
0  2023-09-16  Women's Flyweight          Alexa Grasso  Valentina Shevchenko   
1  2023-09-16       Welterweight  Jack Della Maddalena         Kevin Holland   
2  2023-09-16       Bantamweight        Raul Rosas Jr.     Terrence Mitchell   
3  2023-09-16        Lightweight      Daniel Zellhuber       Christos Giagos   
4  2023-09-16      Featherweight           Kyle Nelson      Fernando Padilla   

    outcome  
0      Draw  
1  fighter1  
2  fighter1  
3  fighter1  
4  fighter1  
Index(['event_date', 'weight_class', 'fighter1', 'fighter2', 'outcome'], dtype='str')


ValueError: empty vocabulary; perhaps the documents only contain stop words